# FarmSense: Agricultural Intelligence Assistant for the Sahel

Gemma 4 Good Hackathon | Kaggle x Google DeepMind

This notebook launches the FarmSense application. It loads the CNN image classifier and the fine-tuned Gemma 4 language model, starts the inference server, and exposes the web interface via a public ngrok tunnel.

Prerequisites:
- GPU T4 must be enabled in the notebook settings (Accelerator: GPU T4 x1)
- Internet access must be enabled
- The two datasets must be attached: PlantVillage (abdallahalidev) and Groundnut Leaf Data (warcoder)

Before running, replace the placeholder values in Cell 1 with your actual tokens.

Execution order: Cell 1, Cell 2, Cell 3, Cell 4, Cell 5.

In [ ]:
# Cell 1: Configuration
#
# Before running this notebook, two steps are required.
#
# Step 1: Attach the datasets.
# In the Kaggle notebook editor, click Add data on the right panel
# and search for the following two datasets. Add both before running.
#   PlantVillage Dataset by abdallahalidev
#   kaggle.com/datasets/abdallahalidev/plantvillage-dataset
#
#   Groundnut Plant Leaf Data by warcoder
#   kaggle.com/datasets/warcoder/groundnut-plant-leaf-data
#
# Step 2: Replace the placeholder tokens below with your actual credentials.
# Do not commit this notebook with real tokens to a public repository.

# HuggingFace token with read access.
# Generate one at: huggingface.co/settings/tokens
HF_TOKEN = "REPLACE_WITH_YOUR_HUGGINGFACE_TOKEN"

# ngrok authentication token.
# Get one at: dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "REPLACE_WITH_YOUR_NGROK_TOKEN"

# GitHub repository URL for the FarmSense application code.
GITHUB_URL = "https://github.com/ndaosaer/farmsense"

# HuggingFace model identifiers.
LLM_MODEL = "ndaosaer/farmsense-gemma4-v2"
CNN_REPO  = "ndaosaer/farmsense-cnn"
CNN_FILE  = "farmsense_cnn.pth"

print('Configuration loaded.')


In [ ]:
# Cell 2: Install dependencies and clone the repository

import os

os.system('pip install -q "huggingface_hub==0.23.4"')
os.system('pip install -q unsloth flask gtts Pillow requests pyngrok torch torchvision')
print('Packages installed.')

if not os.path.exists('/kaggle/working/farmsense'):
    os.system(f'git clone {GITHUB_URL} /kaggle/working/farmsense')
    print('Repository cloned.')
else:
    os.system('git -C /kaggle/working/farmsense pull')
    print('Repository updated.')

# Verify the application structure
required = [
    'app/app_flask.py',
    'app/tools.py',
    'app/templates/index.html',
    'data/diseases.json',
]
for path in required:
    full = f'/kaggle/working/farmsense/{path}'
    status = 'OK' if os.path.exists(full) else 'MISSING'
    print(f'  {status}  {path}')

In [ ]:
# Cell 3: Load models
# This cell loads two models into memory:
#   1. EfficientNet-B0 CNN classifier (99.4% accuracy on 20 crop disease classes)
#   2. Gemma 4 E4B fine-tuned language model for French and Wolof agricultural responses
# Expected duration: 5 to 8 minutes depending on download speed.

import os, torch, gc
import torch.nn as nn
from torchvision import models
from huggingface_hub import hf_hub_download, login

gc.collect()
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

login(token=HF_TOKEN)

# Load the CNN classifier
print('Downloading CNN classifier...')
cnn_path = hf_hub_download(
    repo_id=CNN_REPO,
    filename=CNN_FILE,
    local_dir='/tmp',
    token=HF_TOKEN
)

CLASSES = [
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy', 'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___healthy', 'early_leaf_spot_1', 'late_leaf_spot_1',
    'rust_1', 'early_rust_1', 'nutrition_deficiency_1', 'healthy_leaf_1',
]
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASSES)}

checkpoint = torch.load(cnn_path, map_location='cuda')
model_cnn = models.efficientnet_b0(weights=None)
model_cnn.classifier[1] = nn.Linear(
    model_cnn.classifier[1].in_features, len(CLASSES)
)
model_cnn.load_state_dict(checkpoint['model_state_dict'])
model_cnn = model_cnn.cuda().eval()
print(f'CNN loaded: {len(CLASSES)} classes, 99.4% validation accuracy.')

# Load the fine-tuned language model
print('Loading Gemma 4 fine-tuned model (this may take several minutes)...')
from unsloth import FastLanguageModel

llm_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LLM_MODEL,
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
    device_map={'': 0},
    token=HF_TOKEN,
)
llm_model.eval()
FastLanguageModel.for_inference(llm_model)
inner_tokenizer = tokenizer.tokenizer

print(
    f'Gemma 4 loaded. '
    f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB'
)

In [ ]:
# Cell 4: Start the inference server on port 8000
# This server holds the language model in GPU memory and responds
# to HTTP requests from the Flask application.

import threading, time, torch, requests as req_lib
from flask import Flask as FlaskInfer, request as infer_req, jsonify

SYSTEM_PROMPT = (
    "Tu es FarmSense, un assistant agricole pour les petits agriculteurs "
    "du Sénégal et du Sahel. Tu parles Français et Wolof. Tu es direct et pratique. "
    "REGLES: Jamais de markdown. Texte simple. Maximum 8 lignes. "
    "Pour les maladies: diagnostic, cause, actions numerotees, action immediate. "
    "Pour les prix: prix actuel, tendance, conseils vente, action immediate. "
    "Reponds dans la langue du contexte."
)

infer_app = FlaskInfer('infer')

@infer_app.route('/infer', methods=['POST'])
def infer():
    prompt = infer_req.get_json().get('prompt', '')
    try:
        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
            {'role': 'user',   'content': [{'type': 'text', 'text': prompt}]}
        ]
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors='pt'
        ).to('cuda')
        with torch.no_grad():
            outputs = llm_model.generate(
                input_ids=inputs,
                max_new_tokens=250,
                temperature=0.2,
                do_sample=True,
                pad_token_id=inner_tokenizer.eos_token_id,
            )
        response = inner_tokenizer.decode(
            outputs[0][inputs.shape[1]:],
            skip_special_tokens=True
        ).strip()
        return jsonify({'response': response})
    except Exception as e:
        return jsonify({'error': str(e)})

threading.Thread(
    target=lambda: infer_app.run(
        host='0.0.0.0', port=8000, debug=False, use_reloader=False
    ),
    daemon=True
).start()

time.sleep(3)

# Verify the inference server is responding
r = req_lib.post(
    'http://localhost:8000/infer',
    json={'prompt': '[Zone: Kaolack | Langue: Français]\nPrix de l\'arachide ?'},
    timeout=60
)
print('Inference server status:', 'OK' if r.status_code == 200 else 'ERROR')
print('Test response:', r.json().get('response', r.json())[:100])

In [ ]:
# Cell 5: Launch FarmSense and expose via ngrok
# This cell starts the Flask web application and creates a public tunnel.
# The public URL will be printed at the end of this cell.

import os, sys, subprocess, time
from pyngrok import ngrok, conf

# Set environment variables for the Flask application
os.environ['CNN_PATH']  = '/tmp/farmsense_cnn.pth'
os.environ['INFER_URL'] = 'http://localhost:8000/infer'

# Make the CNN classifier accessible to the Flask application
import pickle, torch
torch.save(
    {
        'model_state_dict': model_cnn.state_dict(),
        'classes':          CLASSES,
        'idx_to_class':     IDX_TO_CLASS,
    },
    '/tmp/farmsense_cnn.pth'
)

# Stop any existing Flask processes
os.system('pkill -f app_flask 2>/dev/null')
time.sleep(2)

# Create the ngrok tunnel
conf.get_default().auth_token = NGROK_TOKEN
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)
time.sleep(2)
public_url = ngrok.connect(5000)

# Start the Flask application
subprocess.Popen(
    ['python', '/kaggle/working/farmsense/app/app_flask.py'],
    env={**os.environ, 'PYTHONPATH': '/kaggle/working/farmsense/app'}
)
time.sleep(6)

# Verify the application is responding
import requests
try:
    r = requests.get('http://localhost:5000/status', timeout=5)
    status = r.json()
    print('Application status:', status.get('message', 'unknown'))
except Exception as e:
    print('Flask did not respond:', e)

print()
print('FarmSense is running.')
print('Public URL:', public_url)
print()
print(
    'Open the URL above in a browser to use the application. '
    'The ngrok tunnel remains active as long as this notebook session is running.'
)